In [ ]:
# Script to write .srt subtitle files for timelapse videos, with timestamps based on file modification date and timelapse speed
# First a function is provided to write a subtitle for a single video. Examples are given of how to call the function for a single video or for all videos in a folder
# Next, code is provided to write a single subtitle file for multiple to-be-combined videos based. This is useful for 
# GoPro timelapses split into multiple short videos, that will later be joined into a single long video. 

from pathlib import Path
from datetime import datetime, timezone, timedelta
import cv2 # requires package opencv. Use e.g. conda install -c conda-forge opencv
import math

In [ ]:
# Helper functions
# Convert seconds to SRT timestamp format
def srt_timestamp(seconds):
    ms_total = int(round(seconds * 1000))
    h, rem = divmod(ms_total, 3600_000)
    m, rem = divmod(rem, 60_000)
    s, ms = divmod(rem, 1000)
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"

In [ ]:
# FUNCTION DEFINITION: WRITE SUBTITLE FOR A SINGLE VIDEO FILE
def create_subtitle_file(video_file, step_real_s=2.0, photos_per_sec=2.0):
    """
    Create an SRT subtitle file for a single timelapse video with real-time timestamps.
    
    Parameters:
    - video_file: path to the input video file (str or Path)
    - step_real_s: subtitle update interval in real time, in seconds (i.e. every step_real_s × photos_per_sec frames). Default: 2 seconds
    - photos_per_sec: number of photos per second taken for the the timelapse, default 2.0
    """
    # Read video file modification time
    p = Path(video_file)
    mtime = datetime.fromtimestamp(p.stat().st_mtime)
    
    # Read video duration using OpenCV
    cap = cv2.VideoCapture(str(video_file))
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open {video_file}")
    
    fps = cap.get(cv2.CAP_PROP_FPS)
    frames = cap.get(cv2.CAP_PROP_FRAME_COUNT)
    dt_sec_video = frames / fps if fps and fps > 0 else float('nan')
    cap.release()
    
    # Calculate real duration and start time
    dt_sec_real = dt_sec_video * fps / photos_per_sec
    t_start = mtime - timedelta(seconds=dt_sec_real)
    
    # Map video-time to real-time
    ratio = dt_sec_real / dt_sec_video if dt_sec_video > 0 else 0.0
    
    def real_time_at(video_seconds):
        return t_start + timedelta(seconds=video_seconds * ratio)
       
    # Convert real-time step to video-time step
    step_video_s = step_real_s / ratio if ratio > 0 else 1.0
    n_cues = max(1, math.ceil(dt_sec_video / step_video_s))
    
    # Write SRT file
    srt_path = Path(video_file).with_suffix(".srt")
    # if srt file exists, remove and give warning
    if srt_path.exists():
        srt_path.unlink()
        print(f"Warning: removed existing SRT file at: {srt_path}")
    with srt_path.open("w", encoding="utf-8") as f:
        for i in range(n_cues):
            t0_v = i * step_video_s
            t1_v = min((i + 1) * step_video_s, dt_sec_video)
            # Text shows the mapped real-world timestamp (use start of cue)
            text = real_time_at(t0_v-8.0/15).strftime("%Y-%m-%d %H:%M:%S")
            
            f.write(f"{i+1}\n")
            f.write(f"{srt_timestamp(t0_v)} --> {srt_timestamp(t1_v)}\n")
            f.write(f"{text}\n\n")
    
    print(f"Wrote subtitles to: {srt_path}")

In [ ]:
# EXAMPLE 1: WRITE SUBTITLE FOR A SINGLE VIDEO
video_file = r'O:\test\in_short\GX010355.mp4'
create_subtitle_file(video_file, step_real_s=2.0, photos_per_sec=2.0) # call function

# EXAMPLE 2: WRITE SUBTITLES FOR ALL VIDEOS IN A FOLDER
step_real_s=2.0
photos_per_sec=2.0
input_folder = r'E:\2025-01-01, Storm 3\GoPros\GoPro4'

# find all mp4 files in the input folder, including subfolders
video_files = list(Path(input_folder).rglob("*.mp4"))

# Call the subtitle creation function for each video file
for video_file in video_files:
    create_subtitle_file(video_file, step_real_s, photos_per_sec)

In [ ]:
# CREATE A SINGLE SUBTITLE FILE FOR MULTIPLE VIDEOS IN A FOLDER, WITH CUMULATIVE TIMING
folder_in = Path(r'O:\test\in_short')
step_real_s = 2.0  # subtitle update interval in real time
video_title = 'S1 Dike-in-dune, Storm 1'

def get_video_info(video_path):
    """Returns (mtime, fps, frames, dt_sec_video, dt_sec_real, t_start)"""
    p = Path(video_path)
    mtime = datetime.fromtimestamp(p.stat().st_mtime)
    
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open {video_path}")
    
    fps = cap.get(cv2.CAP_PROP_FPS)
    frames = cap.get(cv2.CAP_PROP_FRAME_COUNT)
    dt_sec_video = frames / fps if fps and fps > 0 else 0.0
    cap.release()
    
    dt_sec_real = dt_sec_video * fps / 2  # adjust for timelapse
    t_start = mtime - timedelta(seconds=dt_sec_real)
    
    return mtime, fps, frames, dt_sec_video, dt_sec_real, t_start

# Collect all video files (sorted by name)
video_files = sorted(folder_in.glob("*.mp4"))
if not video_files:
    raise RuntimeError(f"No .mp4 files found in {folder_in}")

# Open single subtitle file for all videos
srt_path = folder_in / "merged_subtitles.srt"
cue_index = 1
video_offset = 0.0  # cumulative video time offset

with srt_path.open("w", encoding="utf-8") as f:
    for video_file in video_files:
        print(f"Processing {video_file.name}...")
        mtime, fps, frames, dt_sec_video, dt_sec_real, t_start = get_video_info(video_file)
        
        ratio = dt_sec_real / dt_sec_video if dt_sec_video > 0 else 0.0
        def real_time_at(video_seconds):
            return t_start + timedelta(seconds=video_seconds * ratio)
        
        step_video_s = step_real_s / ratio if ratio > 0 else 1.0
        n_cues = max(1, math.ceil(dt_sec_video / step_video_s))
        
        for i in range(n_cues):
            t0_v = i * step_video_s
            t1_v = min((i + 1) * step_video_s, dt_sec_video)
            
            # Offset by cumulative video time from previous files
            t0_merged = video_offset + t0_v
            t1_merged = video_offset + t1_v
            
            text = real_time_at(t0_v).strftime("%Y-%m-%d %H:%M:%S")
            
            # write title at top of screen
            f.write(f"{cue_index}\n")
            f.write(f"{srt_timestamp(t0_merged)} --> {srt_timestamp(t1_merged)}\n")
            f.write(f"{{\\an7}} {video_title}\n\n")

            # Write timestamp at bottom
            f.write(f"{cue_index+1}\n")
            f.write(f"{srt_timestamp(t0_merged)} --> {srt_timestamp(t1_merged)}\n")
            f.write(f"{text}\n\n")
            
            cue_index += 2
        
        video_offset += dt_sec_video

print(f"Wrote merged subtitles to: {srt_path}")
print(f"Total cues: {cue_index - 1}, Total video duration: {video_offset:.1f}s")